# csv-stream-diff Demo

This notebook installs the published `csv-stream-diff` package from PyPI, generates a small pair of CSV files, runs the comparison, and previews the results.

PyPI: https://pypi.org/project/csv-stream-diff/

In [2]:
%pip install -q csv-stream-diff

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Create a working folder and sample CSV inputs

In [1]:
from pathlib import Path
import csv
import json
import shutil
import subprocess
import sys
import textwrap

import yaml

demo_dir = Path.cwd() / "notebook-demo"
if demo_dir.exists():
    shutil.rmtree(demo_dir)

input_dir = demo_dir / "input"
output_dir = demo_dir / "output"
temp_dir = demo_dir / "tmp"
input_dir.mkdir(parents=True)
output_dir.mkdir(parents=True)
temp_dir.mkdir(parents=True)

left_csv = input_dir / "left.csv"
right_csv = input_dir / "right.csv"
config_path = demo_dir / "config.yaml"

left_rows = [
    {"customer_id": "C001", "transaction_date": "2026-03-20", "amount": "10.00", "status": "OPEN", "description": "alpha"},
    {"customer_id": "C002", "transaction_date": "2026-03-20", "amount": "20.00", "status": "OPEN", "description": "beta"},
    {"customer_id": "C003", "transaction_date": "2026-03-21", "amount": "30.00", "status": "CLOSED", "description": "gamma"},
    {"customer_id": "C004", "transaction_date": "2026-03-21", "amount": "40.00", "status": "OPEN", "description": "delta"},
]

right_rows = [
    {"cust_id": "C001", "txn_dt": "2026-03-20", "transaction_amount": "10.00", "txn_status": "OPEN", "desc": "alpha"},
    {"cust_id": "C002", "txn_dt": "2026-03-20", "transaction_amount": "25.00", "txn_status": "OPEN", "desc": "beta-updated"},
    {"cust_id": "C003", "txn_dt": "2026-03-21", "transaction_amount": "30.00", "txn_status": "CLOSED", "desc": "gamma"},
    {"cust_id": "C005", "txn_dt": "2026-03-22", "transaction_amount": "50.00", "txn_status": "OPEN", "desc": "epsilon"},
]

with left_csv.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=["customer_id", "transaction_date", "amount", "status", "description"])
    writer.writeheader()
    writer.writerows(left_rows)

with right_csv.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=["cust_id", "txn_dt", "transaction_amount", "txn_status", "desc"])
    writer.writeheader()
    writer.writerows(right_rows)

config = {
    "files": {"left": str(left_csv), "right": str(right_csv)},
    "keys": {
        "left": ["customer_id", "transaction_date"],
        "right": ["cust_id", "txn_dt"],
    },
    "compare": {
        "left": ["amount", "status", "description"],
        "right": ["transaction_amount", "txn_status", "desc"],
    },
    "comparison": {
        "case_insensitive": False,
        "trim_whitespace": True,
        "treat_null_as_equal": True,
        "normalize_numeric_values": True,
        "treat_null_as_zero_for_numeric": True,
    },
    "sampling": {"size": 0, "seed": 42},
    "performance": {
        "chunk_size": 2,
        "workers": 1,
        "bucket_count": 4,
        "report_every_rows": 1,
        "temp_directory": str(temp_dir),
        "keep_temp_files": False,
        "show_progress": False,
    },
    "output": {
        "directory": str(output_dir),
        "prefix": "demo_",
        "include_full_rows": True,
        "summary_format": "both",
    },
}

config_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")

print(f"Demo files created in: {demo_dir}")
print(config_path.read_text(encoding='utf-8'))

Demo files created in: C:\repo\csv-stream-diff\notebook-demo
files:
  left: C:\repo\csv-stream-diff\notebook-demo\input\left.csv
  right: C:\repo\csv-stream-diff\notebook-demo\input\right.csv
keys:
  left:
  - customer_id
  - transaction_date
  right:
  - cust_id
  - txn_dt
compare:
  left:
  - amount
  - status
  - description
  right:
  - transaction_amount
  - txn_status
  - desc
comparison:
  case_insensitive: false
  trim_whitespace: true
  treat_null_as_equal: false
sampling:
  size: 0
  seed: 42
performance:
  chunk_size: 2
  workers: 1
  bucket_count: 4
  report_every_rows: 1
  temp_directory: C:\repo\csv-stream-diff\notebook-demo\tmp
  keep_temp_files: false
  show_progress: false
output:
  directory: C:\repo\csv-stream-diff\notebook-demo\output
  prefix: demo_
  include_full_rows: true
  summary_format: both



## 2. Run the comparison

In [4]:
cmd = [sys.executable, "-m", "csvstreamdiff.cli", "--config", str(config_path)]
result = subprocess.run(cmd, capture_output=True, text=True, check=True)

print("STDOUT:")
print(result.stdout)
if result.stderr.strip():
    print("STDERR:")
    print(result.stderr)

STDOUT:
{
  "files": {
    "left": "C:\\repo\\csv-stream-diff\\notebook-demo\\input\\left.csv",
    "right": "C:\\repo\\csv-stream-diff\\notebook-demo\\input\\right.csv"
  },
  "counts": {
    "total_left_rows": 4,
    "total_right_rows": 4,
    "left_unique_keys": 4,
    "right_unique_keys": 4,
    "matched": 3,
    "only_in_left": 1,
    "only_in_right": 1,
    "different_rows": 1,
    "different_cells": 2,
    "duplicate_keys_left": 0,
    "duplicate_keys_right": 0
  },
  "sampling": {
    "mode": "full",
    "requested_size": 0,
    "actual_size": 0,
    "seed": 42,
    "left_unique_population": 4
  },
  "outputs": {
    "only_in_left": "C:\\repo\\csv-stream-diff\\notebook-demo\\output\\demo_only_in_left.csv",
    "only_in_right": "C:\\repo\\csv-stream-diff\\notebook-demo\\output\\demo_only_in_right.csv",
    "differences": "C:\\repo\\csv-stream-diff\\notebook-demo\\output\\demo_differences.csv",
    "duplicate_keys": "C:\\repo\\csv-stream-diff\\notebook-demo\\output\\demo_duplicat

## 3. Load the JSON summary

In [7]:
summary_path = output_dir / "demo_summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8"))
summary

{'files': {'left': 'C:\\repo\\csv-stream-diff\\notebook-demo\\input\\left.csv',
  'right': 'C:\\repo\\csv-stream-diff\\notebook-demo\\input\\right.csv'},
 'counts': {'total_left_rows': 4,
  'total_right_rows': 4,
  'left_unique_keys': 4,
  'right_unique_keys': 4,
  'matched': 3,
  'only_in_left': 1,
  'only_in_right': 1,
  'different_rows': 1,
  'different_cells': 2,
  'duplicate_keys_left': 0,
  'duplicate_keys_right': 0},
 'sampling': {'mode': 'full',
  'requested_size': 0,
  'actual_size': 0,
  'seed': 42,
  'left_unique_population': 4},
 'outputs': {'only_in_left': 'C:\\repo\\csv-stream-diff\\notebook-demo\\output\\demo_only_in_left.csv',
  'only_in_right': 'C:\\repo\\csv-stream-diff\\notebook-demo\\output\\demo_only_in_right.csv',
  'differences': 'C:\\repo\\csv-stream-diff\\notebook-demo\\output\\demo_differences.csv',
  'duplicate_keys': 'C:\\repo\\csv-stream-diff\\notebook-demo\\output\\demo_duplicate_keys.csv',
  'summary_json': 'C:\\repo\\csv-stream-diff\\notebook-demo\\outpu

## 4. Preview the generated output files

In [9]:
def preview_csv(path: Path, max_rows: int = 10):
    with path.open("r", encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle))
    print(f"\n{path.name} ({len(rows)} rows)")
    for row in rows[:max_rows]:
        print(row)

preview_csv(output_dir / "demo_only_in_left.csv")
preview_csv(output_dir / "demo_only_in_right.csv")
preview_csv(output_dir / "demo_differences.csv")
preview_csv(output_dir / "demo_duplicate_keys.csv")


demo_only_in_left.csv (1 rows)
{'key_customer_id': 'C004', 'key_transaction_date': '2026-03-21', 'left_row_json': '{"amount": "40.00", "customer_id": "C004", "description": "delta", "status": "OPEN", "transaction_date": "2026-03-21"}'}

demo_only_in_right.csv (1 rows)
{'key_customer_id': 'C005', 'key_transaction_date': '2026-03-22', 'right_row_json': '{"cust_id": "C005", "desc": "epsilon", "transaction_amount": "50.00", "txn_dt": "2026-03-22", "txn_status": "OPEN"}'}

demo_differences.csv (2 rows)
{'key_customer_id': 'C002', 'key_transaction_date': '2026-03-20', 'left_column': 'amount', 'right_column': 'transaction_amount', 'left_value': '20.00', 'right_value': '25.00', 'left_row_json': '{"amount": "20.00", "customer_id": "C002", "description": "beta", "status": "OPEN", "transaction_date": "2026-03-20"}', 'right_row_json': '{"cust_id": "C002", "desc": "beta-updated", "transaction_amount": "25.00", "txn_dt": "2026-03-20", "txn_status": "OPEN"}'}
{'key_customer_id': 'C002', 'key_transac

## 5. Key takeaways from this demo

- `C004` exists only on the left
- `C005` exists only on the right
- `C002` exists on both sides but differs in `amount` and `description`
- there are no duplicate keys in this sample run

For a heavier demo, increase the input size and turn `performance.show_progress` on.